# Part 10: Convolutional Neural Networks

**Course:** 2026 KMITL Data Analytics

Images have local structure: nearby pixels often describe the same edge or object part. A convolutional neural network (CNN) uses this structure rather than connecting every pixel independently from the start. We inspect filters, pooling and augmentation, train a small clothing classifier, and explain the ideas behind VGG and ResNet.


## Learning objectives

1. Read image height, width and channels and explain the cost of early flattening.
2. Calculate a filter response and describe feature maps, stride, padding and ReLU.
3. Compare max pooling with average pooling.
4. Apply label-preserving image augmentation only during training.
5. Train a CNN, read its history and inspect correct and incorrect predictions.
6. Explain VGG-style blocks and ResNet residual connections without downloading large models.


## Required imports

TensorFlow and Keras provide image operations and training. NumPy supports hand calculations and Matplotlib displays images and curves. Fashion-MNIST downloads on the first load and is reused from the Keras cache afterward. This notebook is independent of Part 9: run every cell in order in a fresh runtime.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)
print('TensorFlow:', tf.__version__)


## Dataset introduction: images are arrays

Fashion-MNIST has ten clothing classes. Each image has height 28, width 28 and one grayscale channel. An RGB color image would have three channels. Keras convolutions use the shape **(batch, height, width, channels)** by default. We add an explicit final dimension for the grayscale channel.

A 2 × 2 grayscale image such as [[0, 255], [128, 64]] has four numbers. After division by 255 they become approximately [[0, 1], [0.502, 0.251]]. In general, `scaled_pixel = pixel / 255`; this fixed operation does not fit statistics to the test data.

For a short CPU run, select 4,000 training images and 800 validation images from the original training split, then 800 images from the separate test split. Class proportions are preserved. The first load downloads about 30 MB; later loads use the cache. Scores from this small subset should not be compared with published full-dataset benchmarks or Part 9's different split sizes.


In [ ]:
(all_train_images, all_train_labels), (all_test_images, all_test_labels) = keras.datasets.fashion_mnist.load_data()
train_indices, validation_indices = train_test_split(np.arange(len(all_train_images)),
    train_size=4000, test_size=800, stratify=all_train_labels, random_state=RANDOM_STATE)
_, test_indices = train_test_split(np.arange(len(all_test_images)), test_size=800,
                                   stratify=all_test_labels, random_state=RANDOM_STATE)
train_images = all_train_images[train_indices].astype('float32')[..., np.newaxis] / 255.0
validation_images = all_train_images[validation_indices].astype('float32')[..., np.newaxis] / 255.0
test_images = all_test_images[test_indices].astype('float32')[..., np.newaxis] / 255.0
train_labels = all_train_labels[train_indices]
validation_labels = all_train_labels[validation_indices]
test_labels = all_test_labels[test_indices]
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
assert set(train_indices).isdisjoint(validation_indices)
assert train_images.shape == (4000, 28, 28, 1)
print('Image batch shape:', train_images.shape)
print('Top-left 4 by 4 pixels of one normalized image:')
print(train_images[0, :4, :4, 0])
fig, axes = plt.subplots(1, 5, figsize=(11, 3))
for index, ax in enumerate(axes):
    ax.imshow(train_images[index, :, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set(title=class_names[train_labels[index]], xlabel='Column', ylabel='Row')
    ax.set_xticks([0, 27])
    ax.set_yticks([0, 27])
fig.suptitle('Clothing images and their training labels')
plt.tight_layout()
plt.show()


**Read the image panels:** brighter pixels have larger values. Rows and columns keep their spatial meaning; the channel axis contains one intensity per pixel. The top-left printed block may be mostly background zeros. An image label describes the whole image, not the value of any one pixel.


## 1. Filters, dot products and feature maps

A **filter**, also called a **kernel**, is a small array of weights that slides over an image. At each location, multiply corresponding pixel and kernel values and add them: this is a dot product. Repeating it gives a **feature map**. The same weights are reused at every location.

Use a 3 × 3 patch whose rows are [0, 0, 1] and a filter whose rows are [−1, 0, 1]. Each row contributes 0 × (−1) + 0 × 0 + 1 × 1 = 1, so the total response is **3**. Adding a zero bias leaves 3. For patch values x_j, filter weights w_j and bias b, `response = sum(x_j * w_j) + b`, where j runs over all patch positions and input channels.

Deep-learning libraries usually call this operation convolution even though they do not flip the kernel; mathematically it is cross-correlation. Our fixed filter highlights vertical intensity changes. In a CNN, training learns filter weights instead of us choosing them. **ReLU** then changes a negative response such as −3 to 0 and keeps 3 as 3: `ReLU(value) = max(0, value)`.

**Stride** is the number of pixels moved each step. **Padding** adds values around the image border. For a 5-pixel dimension, a 3-pixel kernel, no padding and stride 1, the output has 3 positions. With stride 2, it has 2 positions. For input size H, kernel size K, padding P on each side and stride S (without dilation), `output_size = floor((H + 2*P - K) / S) + 1`, where floor rounds down. `valid` adds no padding; TensorFlow's `same` keeps size for stride 1 and uses ceiling division for larger strides, sometimes with unequal padding on the two sides.


In [ ]:
small_image = np.array([[0, 0, 1, 1, 0], [0, 0, 1, 1, 0], [0, 0, 1, 1, 0],
                        [0, 0, 1, 1, 0], [0, 0, 0, 0, 0]], dtype='float32')
kernel = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]], dtype='float32')
image_batch = small_image[np.newaxis, ..., np.newaxis]
kernel_tensor = kernel[..., np.newaxis, np.newaxis]
feature_map = tf.nn.conv2d(image_batch, kernel_tensor, strides=1, padding='VALID').numpy()[0, :, :, 0]
same_map = tf.nn.conv2d(image_batch, kernel_tensor, strides=1, padding='SAME').numpy()
stride_map = tf.nn.conv2d(image_batch, kernel_tensor, strides=2, padding='VALID').numpy()
assert np.isclose((small_image[:3, :3] * kernel).sum(), 3)
assert np.isclose(feature_map[0, 0], 3)
assert feature_map.shape == (3, 3) and same_map.shape == (1, 5, 5, 1)
assert stride_map.shape == (1, 2, 2, 1)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, values, title in zip(axes, [small_image, kernel, feature_map, np.maximum(0, feature_map)],
                            ['Input image', 'Vertical-change filter', 'Feature map', 'After ReLU']):
    plot = ax.imshow(values, cmap='coolwarm')
    for (row, column), value in np.ndenumerate(values):
        ax.text(column, row, f'{value:.0f}', ha='center', va='center', fontsize=9)
    ax.set(title=title, xlabel='Column', ylabel='Row')
    fig.colorbar(plot, ax=ax, shrink=0.65, label='Value')
plt.tight_layout()
plt.show()


**Read the maps:** positive responses identify one direction of intensity change and negative values identify the opposite direction. ReLU removes the negative responses. The feature map is smaller because the 3 × 3 filter must fit inside the image with `valid` padding. Each panel has its own labeled color scale; use the printed numbers for exact comparisons. A CNN needs several learned filters to represent different patterns.


## 2. Max pooling and average pooling

Pooling summarizes small neighborhoods, reducing spatial size. A 2 × 2 block [[1, 2], [3, 4]] becomes **4** with max pooling and (1 + 2 + 3 + 4)/4 = **2.5** with average pooling.

For a region R with n entries, `max_pool(R) = max(R)` and `average_pool(R) = sum(R) / n`. Max pooling keeps the strongest response; average pooling keeps a local mean. Neither has learned weights. Both lose detail, and neither guarantees full invariance to movement. With window 2 and stride 2, a 4 × 4 map becomes 2 × 2.


In [ ]:
pool_input = np.array([[1, 2, 0, 2], [3, 4, 2, 4], [0, 2, 8, 6], [2, 4, 4, 2]], dtype='float32')
pool_batch = pool_input[np.newaxis, ..., np.newaxis]
max_pool = tf.nn.max_pool2d(pool_batch, ksize=2, strides=2, padding='VALID').numpy()[0, :, :, 0]
average_pool = tf.nn.avg_pool2d(pool_batch, ksize=2, strides=2, padding='VALID').numpy()[0, :, :, 0]
assert max_pool[0, 0] == 4 and average_pool[0, 0] == 2.5
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, values, title in zip(axes, [pool_input, max_pool, average_pool], ['Input map', 'Max pooling', 'Average pooling']):
    ax.imshow(values, cmap='Blues', vmin=0, vmax=8)
    for (row, column), value in np.ndenumerate(values):
        ax.text(column, row, f'{value:g}', ha='center', va='center', color='darkred')
    ax.set(title=title, xlabel='Column', ylabel='Row')
plt.tight_layout()
plt.show()


**Read the pooling comparison:** the top-left region becomes 4 or 2.5 depending on the rule. All panels share the same color range, and each cell prints its exact value. Pooling keeps one summary per region, so it cannot reconstruct all original pixels. Max pooling is common after learned feature detectors; average pooling is useful when a distributed response matters.


## 3. Data augmentation

Augmentation creates changed views of training images while keeping their labels. A small rotation imitates a tilted object, translation shifts its position, zoom changes its scale, and horizontal flipping mirrors it. For example, a shift of 10% of a 28-pixel width is about **2.8 pixels**. A Keras rotation factor of 0.04 allows up to 0.04 × 360 = **14.4 degrees** in each direction.

We use mild transformations appropriate for clothing. Large rotations or vertical flips can create unrealistic data; mirroring can change the meaning of text or other datasets. Augmentation is not automatically valid for every task. We display four separate transformations before combining them for training. Keras random preprocessing layers are active during training and inactive during evaluation and prediction. Do not augment validation or test images to make a preferred score.


In [ ]:
example_image = train_images[:1]
transformations = [
    ('Rotation', keras.layers.RandomRotation(0.04, seed=RANDOM_STATE)),
    ('Translation', keras.layers.RandomTranslation(0.08, 0.08, seed=RANDOM_STATE)),
    ('Scaling / zoom', keras.layers.RandomZoom(0.1, seed=RANDOM_STATE)),
    ('Horizontal flip', keras.layers.RandomFlip('horizontal', seed=RANDOM_STATE)),
]
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
axes[0].imshow(example_image[0, :, :, 0], cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Original')
for ax, (name, layer) in zip(axes[1:], transformations):
    changed_image = layer(example_image, training=True).numpy()
    ax.imshow(changed_image[0, :, :, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(name)
    assert changed_image.shape == example_image.shape
for ax in axes:
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'Training views: label stays {class_names[train_labels[0]]}')
plt.tight_layout()
plt.show()


**Read the transformed views:** the object moves, tilts, changes scale or mirrors while retaining its clothing category. RandomFlip can also choose not to flip an image, and near-symmetric objects may look unchanged. Rotation and zoom interpolate pixel values; empty border regions use the layer's default reflection fill. Always inspect augmented examples rather than assuming they are realistic.


## 4. Build the CNN

Flattening a 28 × 28 image before a 64-unit Dense layer creates 784 × 64 + 64 = **50,240 parameters**. A 3 × 3 convolution with one input channel and 16 filters needs only 3 × 3 × 1 × 16 + 16 = **160 parameters**, reused across the image. This is one layer's comparison, not a claim that every CNN has fewer total parameters or less computation.

With kernel height H, width W, input channels C and output filters F, `conv_parameters = H * W * C * F + F`. The final F term is one bias per filter. Convolutions preserve local relationships and share weights, although they still calculate many responses across positions.

Our model applies augmentation, then two Conv2D–ReLU–MaxPooling stages. The shapes change from 28 × 28 × 1 to 28 × 28 × 16, then 14 × 14 × 16, then 14 × 14 × 32 and finally 7 × 7 × 32. Only then do we **Flatten** to 1,568 values, add a small Dense layer and predict ten softmax probabilities. Repeated convolutional filters learn local features before classification.


In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal', seed=RANDOM_STATE),
    keras.layers.RandomRotation(0.04, seed=RANDOM_STATE + 1),
    keras.layers.RandomTranslation(0.08, 0.08, seed=RANDOM_STATE + 2),
    keras.layers.RandomZoom(0.1, seed=RANDOM_STATE + 3),
], name='training_augmentation')
cnn = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    augmentation,
    keras.layers.Conv2D(16, 3, padding='same', activation='relu', name='first_convolution'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(10, activation='softmax'),
], name='small_fashion_cnn')
cnn.compile(optimizer=keras.optimizers.Adam(0.001),
            loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False), metrics=['accuracy'])
cnn.summary()
assert cnn.get_layer('first_convolution').count_params() == 160
assert np.allclose(augmentation(example_image, training=False).numpy(), example_image)


## 5. Train and read the history

Integer class IDs match sparse categorical cross-entropy. For a true class assigned probability 0.5, loss is −ln(0.5) ≈ **0.693**. In general, `loss = -ln(probability_of_true_class)`, averaged across the batch. Adam adjusts parameters to reduce that loss.

We train at most five epochs with batches of 64. Early stopping monitors validation loss and restores the best weights if training stops early or finishes. Test images remain untouched until model selection is complete. The dataset options limit background threads for a small local runtime; augmentation remains inside the model and affects training batches only.


In [ ]:
data_options = tf.data.Options()
data_options.threading.private_threadpool_size = 1
train_data = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_data = train_data.shuffle(len(train_images), seed=RANDOM_STATE).batch(64).with_options(data_options)
validation_data = tf.data.Dataset.from_tensor_slices((validation_images, validation_labels)).batch(64).with_options(data_options)
test_data = tf.data.Dataset.from_tensor_slices((test_images, test_labels)).batch(64).with_options(data_options)
history = cnn.fit(train_data, validation_data=validation_data, epochs=5, shuffle=False, verbose=2,
                  callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)])
epochs = np.arange(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(epochs, history.history[metric], 'o-', label='Training with augmentation')
    ax.plot(epochs, history.history['val_' + metric], 'o-', label='Validation without augmentation')
    ax.set(title=f'CNN {metric}', xlabel='Epoch', ylabel=metric.title(), xticks=epochs)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


**Read the training history:** loss should generally improve, but it need not decrease every epoch. Validation accuracy can exceed training accuracy because training images are randomly changed and training metrics average predictions made while weights are changing. This is not proof of a data leak by itself. A growing validation-loss gap can still warn of overfitting. More epochs or stronger augmentation are not automatically better.


## 6. Inspect both successes and mistakes

Evaluate the selected network once on the held-out test subset. We then show up to four correct and four incorrect examples, using the model's largest class probability as its prediction. For example, probabilities [0.1, 0.2, 0.7] select class index **2**. An incorrect high-confidence result deserves attention: confidence is not certainty. The image panels are selected for diagnosis and are not a representative accuracy sample.


In [ ]:
test_results = cnn.evaluate(test_data, return_dict=True, verbose=0)
print('Held-out test results:', test_results)
prediction_data = tf.data.Dataset.from_tensor_slices(test_images).batch(64).with_options(data_options)
probabilities = cnn.predict(prediction_data, verbose=0)
predicted_labels = probabilities.argmax(axis=1)
correct_indices = np.flatnonzero(predicted_labels == test_labels)
incorrect_indices = np.flatnonzero(predicted_labels != test_labels)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for row, (indices, status) in enumerate([(correct_indices, 'Correct'), (incorrect_indices, 'Incorrect')]):
    for column, ax in enumerate(axes[row]):
        if column >= len(indices):
            ax.text(0.5, 0.5, f'No further {status.lower()} example', ha='center', va='center', wrap=True)
            ax.axis('off')
            continue
        index = indices[column]
        ax.imshow(test_images[index, :, :, 0], cmap='gray', vmin=0, vmax=1)
        actual = class_names[test_labels[index]]
        predicted_name = class_names[predicted_labels[index]]
        confidence = probabilities[index].max()
        ax.set_title(f'{status}: true {actual}\nPred: {predicted_name} ({confidence:.2f})', fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle('Selected correct and incorrect test predictions')
plt.tight_layout()
plt.show()


**Read the two rows:** the first contains correct cases; the second contains mistakes if any exist. Titles show the actual class, predicted class and its probability. Similar tops are often difficult to distinguish in 28 × 28 images, but inspect the actual examples rather than assuming every error has the same cause. The reported test accuracy uses all 800 test images, not only these selected panels. Do not tune against these test errors and then reuse the same set as an untouched final test.


## 7. VGG and ResNet: two architectural ideas

**VGG** popularized repeated small 3 × 3 convolutions, with pooling between blocks and more channels in later blocks. Stacking small filters adds nonlinear transformations while expanding the region each unit can observe. Classic VGG models are much larger than this lesson's model, especially their dense classifier. Our two convolution stages illustrate the idea, not the full VGG architecture.

**ResNet** adds shortcut connections. Instead of making a block learn the whole desired output, it can learn a **residual**, or change to the input. If the input is [1, 2] and the learned change is [0.2, −0.5], adding them gives **[1.2, 1.5]**. In general, for input x and a learned block F, `output = x + F(x)` before any following activation.

The shortcut provides another route for information and gradients, helping optimization in deep networks. It does not guarantee good predictions or remove all training problems. Addition requires matching shapes; when spatial size or channel count changes, a learned projection such as a 1 × 1 convolution can align them. The next diagram uses chosen numbers to explain the addition, not a trained ResNet.


In [ ]:
residual_input = np.array([1., 2.])
learned_change = np.array([0.2, -0.5])
residual_output = residual_input + learned_change
assert np.allclose(residual_output, [1.2, 1.5])
fig, ax = plt.subplots(figsize=(11, 4))
boxes = [(0.12, 0.35, 'Input x: [1, 2]'), (0.48, 0.35, 'Block F(x): [0.2, -0.5]'),
         (0.86, 0.35, 'Add: [1.2, 1.5]')]
for x, y, text in boxes:
    ax.text(x, y, text, ha='center', va='center', transform=ax.transAxes,
            bbox={'boxstyle': 'round,pad=0.7', 'facecolor': 'lightblue'})
for start, end in [(0.25, 0.34), (0.65, 0.74)]:
    ax.annotate('', xy=(end, 0.35), xytext=(start, 0.35), xycoords='axes fraction',
                arrowprops={'arrowstyle': '->', 'lw': 2})
ax.annotate('', xy=(0.86, 0.45), xytext=(0.12, 0.45), xycoords='axes fraction',
            arrowprops={'arrowstyle': '->', 'connectionstyle': 'bar,fraction=-0.14', 'color': 'darkorange', 'lw': 2})
ax.text(0.5, 0.9, 'Identity shortcut: pass x unchanged', color='darkorange', ha='center', transform=ax.transAxes)
ax.set_title('A residual block learns a change, then adds the original input')
ax.axis('off')
plt.tight_layout()
plt.show()


**Read the residual diagram:** the blue path represents a learned transformation and the orange shortcut carries the original input. They meet at addition, not concatenation. A zero learned change would leave the input unchanged. Real residual blocks use learned convolutional operations on feature maps, while this example isolates the arithmetic.


## Common mistakes

- Forgetting the channel dimension or confusing channel count with the number of images.
- Treating a feature map as a final class prediction.
- Assuming `same` padding preserves size for every stride.
- Expecting pooling to learn weights or keep every image detail.
- Applying strong transformations without checking whether labels remain valid.
- Augmenting test data during ordinary evaluation or comparing scores from different subsets as a controlled experiment.
- Adding residual tensors with incompatible shapes, or claiming our small CNN is a full VGG or ResNet.


In [ ]:
assert probabilities.shape == (800, 10)
assert np.isfinite(probabilities).all()
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
assert np.isclose(test_results['accuracy'], np.mean(predicted_labels == test_labels), atol=1e-6)
assert len(correct_indices) + len(incorrect_indices) == len(test_images)
assert all(np.isfinite(history.history[key]).all() for key in ['loss', 'val_loss', 'accuracy', 'val_accuracy'])
assert np.allclose(augmentation(test_images[:2], training=False).numpy(), test_images[:2])
assert np.isfinite(test_results['loss'])
print('CNN lesson checks passed.')


## Summary

Convolutional filters share weights over local image neighborhoods, creating feature maps. Stride and padding control their size; ReLU adds nonlinearity and pooling reduces detail. Appropriate augmentation provides varied training views without changing the evaluation data. Our small CNN completes an image-classification workflow, including error inspection. VGG-style blocks repeat local feature extraction, while ResNet shortcuts make learning changes to representations easier.
